In [1]:
import pandas as pd 

rent = pd.read_csv("../data/processed/clean_rtba_median_rent_suburb.csv")
price = pd.read_csv("../data/processed/clean_median_house_price_suburb.csv")

In [8]:
#Filtering to house rentals only 
house_rent = rent[rent["property_type"] == "3 Bed House"].copy()
house_rent

,region,suburb,median_rent,property_type
161,Inner Melbourne,Albert Park-Middle Park-West St Kilda,1110.0,3 Bed House
162,Inner Melbourne,Armadale,1100.0,3 Bed House
163,Inner Melbourne,Carlton North,930.0,3 Bed House
164,Inner Melbourne,Carlton-Parkville,900.0,3 Bed House
165,Inner Melbourne,CBD-St Kilda Rd,NaN,3 Bed House
...,...,...,...,...
317,Other Regional Centres,Warrnambool,550.0,3 Bed House
318,Other Regional Centres,Wodonga,500.0,3 Bed House
319,Other Regional Centres,Total,465.0,3 Bed House
320,Other Regional Centres,NaN,NaN,3 Bed House


In [11]:
#Checking for missing values in house rent 
house_rent[house_rent["median_rent"].isna()]

,region,suburb,median_rent,property_type
165,Inner Melbourne,CBD-St Kilda Rd,NaN,3 Bed House
167,Inner Melbourne,Docklands,NaN,3 Bed House
180,Inner Melbourne,Southbank,NaN,3 Bed House
236,North Western Melbourne,Keilor,NaN,3 Bed House
320,Other Regional Centres,NaN,NaN,3 Bed House
321,Other Regional Centres,NaN,NaN,3 Bed House


In [13]:
#Dropping 6 missing values 
house_rent = house_rent.dropna(subset=["median_rent"])

In [14]:
house_rent

,region,suburb,median_rent,property_type
161,Inner Melbourne,Albert Park-Middle Park-West St Kilda,1110.0,3 Bed House
162,Inner Melbourne,Armadale,1100.0,3 Bed House
163,Inner Melbourne,Carlton North,930.0,3 Bed House
164,Inner Melbourne,Carlton-Parkville,900.0,3 Bed House
166,Inner Melbourne,Collingwood-Abbotsford,950.0,3 Bed House
...,...,...,...,...
315,Other Regional Centres,Wangaratta,460.0,3 Bed House
316,Other Regional Centres,Warragul,480.0,3 Bed House
317,Other Regional Centres,Warrnambool,550.0,3 Bed House
318,Other Regional Centres,Wodonga,500.0,3 Bed House


In [24]:
#Final check for each dataset before merging 
house_rent[["suburb", "median_rent"]].head()

,suburb,median_rent
161,Albert Park-Middle Park-West St Kilda,1110.0
162,Armadale,1100.0
163,Carlton North,930.0
164,Carlton-Parkville,900.0
166,Collingwood-Abbotsford,950.0


In [22]:
price[["suburb", "median_price", "sales_count"]].head()

,suburb,median_price,sales_count
0,Nan,2025.0,2025
1,Abbotsford,1310000.0,9
2,Aberfeldie,2045000.0,11
3,Aintree,720000.0,39
4,Aireys Inlet,1312500.0,4


In [29]:
#Dropping first row from price data - junk header value from raw excel data 
price = price[price["suburb"].str.lower() != "nan"]

In [30]:
price.head()

,suburb,median_price,sales_count
1,Abbotsford,1310000.0,9
2,Aberfeldie,2045000.0,11
3,Aintree,720000.0,39
4,Aireys Inlet,1312500.0,4
5,Airport West,1000000.0,29


In [31]:
#Checking suburbs names/format 
house_rent["suburb"]

161    Albert Park-Middle Park-West St Kilda
162                                 Armadale
163                            Carlton North
164                        Carlton-Parkville
166                   Collingwood-Abbotsford
                       ...                  
315                               Wangaratta
316                                 Warragul
317                              Warrnambool
318                                  Wodonga
319                                    Total
Name: suburb, Length: 155, dtype: object

In [32]:
price["suburb"]


1        Abbotsford
2        Aberfeldie
3           Aintree
4      Aireys Inlet
5      Airport West
           ...     
747          Yarram
748      Yarraville
749      Yarrawonga
750             Yea
751          Yinnar
Name: suburb, Length: 751, dtype: object

In [40]:
#Standardising suburb names 
def standardise_suburb_basic(s):
    return (
        s.astype(str)
         .str.strip() #for leading spaces
         .str.title() #consistent casing 
    )

In [41]:
house_rent["suburb"] = standardise_suburb_basic(house_rent["suburb"])
price["suburb"] = standardise_suburb_basic(price["suburb"])

C:\Users\trabi\AppData\Local\Temp\ipykernel_43912\2873478430.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  house_rent["suburb"] = standardise_suburb_basic(house_rent["suburb"])


In [42]:
import re 


In [43]:
def standardise_suburb_punct(s):
    return(
        s
        .str.replace(r"[^\w\s]", "", regex=True) #to remove punctuations 
        .str.replace(r"\s+", " ", regex=True) #normalise spaces 
    )

In [44]:
house_rent["suburb"] = standardise_suburb_punct(house_rent["suburb"])
price["suburb"] = standardise_suburb_punct(price["suburb"])

C:\Users\trabi\AppData\Local\Temp\ipykernel_43912\3333482287.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  house_rent["suburb"] = standardise_suburb_punct(house_rent["suburb"])


In [45]:
#Checking unmatched suburbs 
rent_suburbs = set(house_rent["suburb"])
price_suburbs = set(price["suburb"])

In [47]:
missing_in_price = sorted(rent_suburbs - price_suburbs)
missing_in_rent = sorted(price_suburbs - rent_suburbs)

In [52]:
len(missing_in_price), len(missing_in_rent)
#from output, 66 suburbs in rent data not found in price data (small and expected)
#674 suburbs in price data not found in rent data - likely because RTBA data sets cover new lettings and since rent data is restricted to 3-bed houses, this further reduces coverage

(66, 674)

In [49]:
missing_in_price[:15]


['Albert ParkMiddle ParkWest St Kilda',
 'AspendaleChelseaCarrum',
 'Ballarat',
 'BelmontGrovedale',
 'BroadmeadowsRoxburgh Park',
 'BulleenTemplestoweDoncaster',
 'BundooraGreensboroughHurstbridge',
 'BurwoodAshburton',
 'CamberwellGlen Iris',
 'CanterburySurrey HillsMont Albert',
 'CarltonParkville',
 'ChadstoneOakleigh',
 'CoburgPascoe Vale South',
 'CollingwoodAbbotsford',
 'CroydonLilydale']

In [50]:
missing_in_rent[:15]

['Abbotsford',
 'Aberfeldie',
 'Aintree',
 'Aireys Inlet',
 'Airport West',
 'Albanvale',
 'Albert Park',
 'Albion',
 'Alexandra',
 'Alfredton',
 'Allansford',
 'Alphington',
 'Altona East',
 'Altona Meadows',
 'Altona North']

In [53]:
#Using inner join to analyse suburbs with both reliable 3-bed house rent data and sufficient sales activity 

merged = house_rent.merge(price, on="suburb", how="inner")

In [54]:
merged.head()

,region,suburb,median_rent,property_type,median_price,sales_count
0,Inner Melbourne,Armadale,1100.0,3 Bed House,2588000.0,20
1,Inner Melbourne,Carlton North,930.0,3 Bed House,1730000.0,20
2,Inner Melbourne,East Melbourne,1200.0,3 Bed House,1696000.0,2
3,Inner Melbourne,Elwood,1000.0,3 Bed House,2542500.0,20
4,Inner Melbourne,Fitzroy,1100.0,3 Bed House,1600000.0,24


In [56]:
len(house_rent), len(price), len(merged)

(155, 751, 77)

In [57]:
merged[["suburb","median_rent","median_price"]].isna().sum()

suburb          0
median_rent     0
median_price    0
dtype: int64

In [58]:
#Computing gross rental yield 
merged["gross_yield"] = (52* merged["median_rent"])/ merged["median_price"]
merged["gross_yield_pct"] = merged["gross_yield"] * 100

In [59]:
merged["gross_yield_pct"].describe()

count    77.000000
mean      3.396312
std       1.067645
min       1.216129
25%       2.569767
50%       3.471539
75%       4.090377
max       5.925926
Name: gross_yield_pct, dtype: float64

In [62]:
merged.sort_values("gross_yield_pct", ascending=False).head(10)
#high yields are in cheaper suburbs (other regional centres)

,region,suburb,median_rent,property_type,median_price,sales_count,gross_yield,gross_yield_pct
66,Other Regional Centres,Morwell,400.0,3 Bed House,351000.0,96,0.059259,5.925926
59,Other Regional Centres,Bairnsdale,450.0,3 Bed House,450000.0,35,0.052000,5.200000
69,Other Regional Centres,Shepparton,470.0,3 Bed House,470500.0,188,0.051945,5.194474
64,Other Regional Centres,Horsham,410.0,3 Bed House,413100.0,68,0.051610,5.160978
65,Other Regional Centres,Mildura,475.0,3 Bed House,485000.0,165,0.050928,5.092784
60,Other Regional Centres,Benalla,450.0,3 Bed House,465000.0,55,0.050323,5.032258
67,Other Regional Centres,Portland,425.0,3 Bed House,445000.0,57,0.049663,4.966292
68,Other Regional Centres,Seymour,420.0,3 Bed House,443000.0,27,0.049300,4.930023
58,Bendigo,North Bendigo,470.0,3 Bed House,497500.0,23,0.049126,4.912563
75,Other Regional Centres,Warrnambool,550.0,3 Bed House,600000.0,122,0.047667,4.766667


In [63]:
merged.sort_values("gross_yield_pct").head(10)
#lower yields in inner melbourne 

,region,suburb,median_rent,property_type,median_price,sales_count,gross_yield,gross_yield_pct
10,Inner Eastern Melbourne,Balwyn,725.0,3 Bed House,3100000.0,31,0.012161,1.216129
15,Inner Eastern Melbourne,Kew,845.0,3 Bed House,2745000.0,54,0.016007,1.600729
24,Southern Melbourne,Malvern,1050.0,3 Bed House,3340000.0,20,0.016347,1.634731
18,Southern Melbourne,Brighton,1150.0,3 Bed House,3341800.0,58,0.017895,1.789455
9,Inner Melbourne,Toorak,1250.0,3 Bed House,3425000.0,9,0.018978,1.897810
11,Inner Eastern Melbourne,Blackburn,600.0,3 Bed House,1638000.0,28,0.019048,1.904762
25,Southern Melbourne,Malvern East,750.0,3 Bed House,2000000.0,51,0.019500,1.950000
12,Inner Eastern Melbourne,Box Hill,650.0,3 Bed House,1725000.0,7,0.019594,1.959420
14,Inner Eastern Melbourne,Hawthorn,925.0,3 Bed House,2440000.0,26,0.019713,1.971311
19,Southern Melbourne,Brighton East,905.0,3 Bed House,2337500.0,42,0.020133,2.013262


In [64]:
merged.to_csv(
    "../data/processed/suburb_house_rental_yield.csv",
    index=False
)